# 04. Atribuição — mãe + endereço (modelo + resgate)

Lê as predictions do
[`02d_aplicar_splink_mae_endereco.ipynb`](02d_aplicar_splink_mae_endereco.ipynb).

Faixas (cima → baixo). Teto 3 cumulativo: CPF já preenchido acima conta.

| Faixa | Decisão |
|---|---|
| `p ≥ 0,90` | aceita pelo modelo (`score`) |
| `0,50 ≤ p < 0,90` | R1 ou R2 ou R3 |
| `0,25 ≤ p < 0,50` | R1 forte ou R2 ou R3 forte |
| `p < 0,25` | não resgata |

**R1** — data exata + primeiro nome compatível + (nome contido ou cobertura de tokens).
**R2** — nome completo igual/contido + data plausível (≠ exata): |Δdias|≤1, ou mesmo dia/mês com |Δano|∈{1,10}.
**R3** — primeiro compatível + cobertura nominal + mãe presente + JW mãe.

Prioridade no par: R1 > R2 > R3. Empate de CPF: `p`, mãe preenchida, logradouro, CEP.

Exporta `splink_atribuicao_mae_endereco.parquet`.


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    MAX_CENSOS_POR_CPF,
    SPLINK_ATRIBUICAO_MAE_ENDERECO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    export_parquet,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

PISO_MODELO = 0.90
PISO_RESGATE = 0.50
PISO_BAIXO = 0.25
TETO = MAX_CENSOS_POR_CPF

print_paths()
require_input(
    SPLINK_PREDICTIONS_MAE_ENDERECO,
    label='SPLINK_PREDICTIONS_MAE_ENDERECO (rode o 02d antes)',
)
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)
print(
    f'Faixas: modelo>={PISO_MODELO} | resgate>={PISO_RESGATE} | '
    f'baixo>={PISO_BAIXO} | teto={TETO}'
)
print('Predictions:', SPLINK_PREDICTIONS_MAE_ENDERECO)


In [ ]:
_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')

con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS_MAE_ENDERECO}')
''')
n_pred = con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]
print('Pares nas predictions:', f'{n_pred:,}')


## Pares elegíveis ao resgate

Pré-calcula R1/R2/R3 (versão média e baixa) para `p ≥ 0,25`. A faixa
`p ≥ 0,90` não usa essas regras.


In [ ]:
tok_ca = "string_split(ca.nome_completo_phon, ' ')"
tok_pb = "string_split(pb.nome_completo_phon, ' ')"
n_ca = f"len({tok_ca})"
n_pb = f"len({tok_pb})"
n_min = f"least({n_ca}, {n_pb})"
n_inter = f"len(list_intersect({tok_ca}, {tok_pb}))"
cobertura = f"({n_inter} * 1.0 / nullif({n_min}, 0))"
contido = (
    f"{n_min} >= 2 AND {n_ca} <> {n_pb} AND {n_inter} = {n_min}"
)
primeiro_ok = (
    "ca.primeiro_nome_phon IS NOT NULL AND pb.primeiro_nome_phon IS NOT NULL "
    "AND ("
    "ca.primeiro_nome_phon = pb.primeiro_nome_phon "
    "OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) <= 1"
    ")"
)
data_igual = (
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento = pb.data_nascimento"
)
data_ambas = (
    "ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento <> pb.data_nascimento"
)
delta_dias = (
    "abs(date_diff('day', "
    "CAST(ca.data_nascimento AS DATE), CAST(pb.data_nascimento AS DATE)))"
)
mesmo_dia_mes = (
    "day(CAST(ca.data_nascimento AS DATE)) = day(CAST(pb.data_nascimento AS DATE)) "
    "AND month(CAST(ca.data_nascimento AS DATE)) = month(CAST(pb.data_nascimento AS DATE))"
)
delta_ano = (
    "abs(year(CAST(ca.data_nascimento AS DATE)) "
    "- year(CAST(pb.data_nascimento AS DATE)))"
)
data_plausivel = (
    f"({delta_dias} <= 1) OR "
    f"(({mesmo_dia_mes}) AND ({delta_ano}) = 1) OR "
    f"(({mesmo_dia_mes}) AND ({delta_ano}) = 10)"
)
nome_muito_forte = (
    f"(ca.nome_completo_phon = pb.nome_completo_phon) OR ({contido})"
)
mae_presente = (
    "ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL"
)
jw_mae = "jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon)"

r1_medio = (
    f"({data_igual}) AND ({primeiro_ok}) AND "
    f"(({contido}) OR ({cobertura} >= 0.75))"
)
r1_baixo = (
    f"({data_igual}) AND ({primeiro_ok}) AND "
    f"(({contido}) OR ({cobertura} >= 0.80))"
)
r2 = (
    f"({nome_muito_forte}) AND ({data_ambas}) AND ({data_plausivel})"
)
r3_medio = (
    f"({primeiro_ok}) AND ({cobertura} >= 0.55) AND ({mae_presente}) "
    f"AND ({jw_mae} >= 0.75)"
)
r3_baixo = (
    f"({primeiro_ok}) AND ({cobertura} >= 0.75) AND ({mae_presente}) "
    f"AND ({jw_mae} >= 0.85)"
)

desempate = (
    "("
    "CAST(ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL AS INTEGER) "
    "+ CAST("
    "ca.logradouro_norm IS NOT NULL AND pb.logradouro_norm IS NOT NULL "
    "AND ca.logradouro_norm = pb.logradouro_norm AS INTEGER) "
    "+ CAST("
    "ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep AS INTEGER)"
    ")"
)

con.execute(f'''
CREATE OR REPLACE TABLE pares AS
SELECT
    p.unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    {desempate} AS desempate,
    ({r1_medio}) AS r1_medio,
    ({r1_baixo}) AS r1_baixo,
    ({r2}) AS r2,
    ({r3_medio}) AS r3_medio,
    ({r3_baixo}) AS r3_baixo,
    CASE
        WHEN ({r1_medio}) THEN 'R1'
        WHEN ({r2}) THEN 'R2'
        WHEN ({r3_medio}) THEN 'R3'
    END AS regra_medio,
    CASE
        WHEN ({r1_baixo}) THEN 'R1'
        WHEN ({r2}) THEN 'R2'
        WHEN ({r3_baixo}) THEN 'R3'
    END AS regra_baixo
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
WHERE p.match_probability >= {PISO_BAIXO}
''')

n_pares = con.execute('SELECT COUNT(*) FROM pares').fetchone()[0]
print('Pares com p >=', PISO_BAIXO, ':', f'{n_pares:,}')
display(con.execute('''
SELECT
    COUNT(*) FILTER (WHERE r1_medio) AS n_r1_medio,
    COUNT(*) FILTER (WHERE r2) AS n_r2,
    COUNT(*) FILTER (WHERE r3_medio) AS n_r3_medio,
    COUNT(*) FILTER (WHERE r1_baixo) AS n_r1_baixo,
    COUNT(*) FILTER (WHERE r3_baixo) AS n_r3_baixo,
    COUNT(*) FILTER (WHERE regra_medio IS NOT NULL) AS n_com_regra_medio,
    COUNT(*) FILTER (WHERE regra_baixo IS NOT NULL) AS n_com_regra_baixo
FROM pares
''').df())


## Atribuição por faixa

Três passos. Em cada um: Censos livres, CPFs abaixo do teto, melhor CPF por
Censo. No lote, se `n_já + n_novo > teto`, aquele CPF não entra.


In [ ]:
con.execute('''
CREATE OR REPLACE TABLE atribuicao (
    unique_id_censo VARCHAR,
    unique_id_cpf VARCHAR,
    match_probability DOUBLE,
    faixa VARCHAR,
    regra VARCHAR
)
''')

# (nome_faixa, piso, teto_nota ou None, coluna_regra ou None)
faixas = [
    ('modelo', PISO_MODELO, None, None),
    ('resgate', PISO_RESGATE, PISO_MODELO, 'regra_medio'),
    ('baixo', PISO_BAIXO, PISO_RESGATE, 'regra_baixo'),
]

for nome_faixa, piso, teto_nota, col_regra in faixas:
    if teto_nota is None:
        filtro_p = f'match_probability >= {piso}'
    else:
        filtro_p = (
            f'match_probability >= {piso} '
            f'AND match_probability < {teto_nota}'
        )
    if col_regra is None:
        filtro_regra = 'TRUE'
        regra_sql = "'score'"
    else:
        filtro_regra = f'{col_regra} IS NOT NULL'
        regra_sql = col_regra

    con.execute(f'''
    CREATE OR REPLACE TABLE faixa_base AS
    SELECT
        unique_id_censo,
        unique_id_cpf,
        match_probability,
        desempate,
        {regra_sql} AS regra
    FROM pares
    WHERE {filtro_p}
      AND {filtro_regra}
      AND unique_id_censo NOT IN (SELECT unique_id_censo FROM atribuicao)
      AND unique_id_cpf NOT IN (
          SELECT unique_id_cpf
          FROM atribuicao
          GROUP BY 1
          HAVING COUNT(*) >= {TETO}
      )
    ''')

    con.execute('''
    CREATE OR REPLACE TABLE faixa_melhor AS
    SELECT unique_id_censo, unique_id_cpf, match_probability, regra
    FROM faixa_base
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY unique_id_censo
        ORDER BY match_probability DESC, desempate DESC, unique_id_cpf
    ) = 1
    ''')

    antes = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
    con.execute(f'''
    INSERT INTO atribuicao
    SELECT
        m.unique_id_censo,
        m.unique_id_cpf,
        m.match_probability,
        '{nome_faixa}' AS faixa,
        m.regra
    FROM faixa_melhor m
    JOIN (
        SELECT unique_id_cpf, COUNT(*) AS n_novo
        FROM faixa_melhor
        GROUP BY 1
    ) g ON g.unique_id_cpf = m.unique_id_cpf
    LEFT JOIN (
        SELECT unique_id_cpf, COUNT(*) AS n_ja
        FROM atribuicao
        GROUP BY 1
    ) j ON j.unique_id_cpf = m.unique_id_cpf
    WHERE coalesce(j.n_ja, 0) + g.n_novo <= {TETO}
    ''')
    depois = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
    print(f'{nome_faixa} ({filtro_p}): +{depois - antes:,} (total {depois:,})')


In [ ]:
n_app = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''').fetchone()[0]
n_ok = con.execute('SELECT COUNT(*) FROM atribuicao').fetchone()[0]
print(f'Censo aplicação: {n_app:,}')
print(f'Atribuídos: {n_ok:,} ({100.0 * n_ok / n_app:.1f}%)')

display(con.execute('''
SELECT
    faixa,
    regra,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM atribuicao
GROUP BY 1, 2
ORDER BY
    CASE faixa
        WHEN 'modelo' THEN 1
        WHEN 'resgate' THEN 2
        WHEN 'baixo' THEN 3
    END,
    regra
''').df())

display(con.execute('''
SELECT
    n_censos,
    COUNT(*) AS n_cpf,
    n_censos * COUNT(*) AS n_pares
FROM (
    SELECT unique_id_cpf, COUNT(*) AS n_censos
    FROM atribuicao
    GROUP BY 1
)
GROUP BY 1
ORDER BY 1
''').df())


In [ ]:
export_parquet(con, 'atribuicao', path=SPLINK_ATRIBUICAO_MAE_ENDERECO)
print('Exportado:', SPLINK_ATRIBUICAO_MAE_ENDERECO)
con.close()
